In [1]:
import pandas as pd

In [32]:
rows = []
with open('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/MASIF/dpocket/dpocket_out_4A_exp.txt', 'r') as f:
    lines = f.readlines()
    col_names = lines[0].split()
    for line in lines[1:]:
        rows.append(line.split())
df = pd.DataFrame(rows, columns=col_names)
df['pdb'] = df['pdb'].str[-8:-4]
df

,pdb,lig,overlap,PP-crit,PP-dst,crit4,crit5,crit6,crit6_continue,lig_vol,...,LEU,LYS,MET,PHE,PRO,SER,THR,TRP,TYR,VAL
0,5EFQ,ADP,100.00,1,0.00,1.00,1.00,1,2.00,265.10,...,3,2,1,1,0,1,2,0,1,1
1,4CRM,ADP,100.00,1,0.00,1.00,1.00,1,2.00,313.42,...,0,3,0,1,0,2,2,0,0,0
2,3CR3,ADP,100.00,1,0.00,1.00,1.00,1,2.00,559.15,...,1,2,2,0,1,2,1,0,1,1
3,3P4X,ADP,100.00,1,0.00,1.00,1.00,1,2.00,248.16,...,2,2,0,1,0,1,4,0,0,1
4,3KB1,ADP,100.00,1,0.00,1.00,1.00,1,2.00,482.34,...,5,2,1,0,2,3,2,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1440,1UAK,SAM,100.00,1,0.00,1.00,1.00,1,2.00,640.77,...,3,0,0,0,2,3,1,1,3,2
1441,4NJJ,SAM,100.00,1,0.00,1.00,1.00,1,2.00,780.29,...,0,2,1,2,1,2,3,0,0,1
1442,5C8T,SAM,100.00,1,0.00,1.00,1.00,1,2.00,262.18,...,1,1,0,2,1,0,0,2,1,2
1443,2G72,SAM,100.00,1,0.00,1.00,1.00,1,2.00,279.88,...,1,0,0,3,1,1,1,0,4,3


In [33]:
df[['pdb','lig']].nunique()

pdb    1416
lig       7
dtype: int64

In [ ]:
cols_with_diff_values = []
for col in df.columns:
    if col == 'pdb' or col == 'lig':
        continue
    if df[col].nunique() > 1:
        cols_with_diff_values.append(col)
print("num cols with diff values", len(cols_with_diff_values))

num cols with diff values 41


In [65]:
cols_with_diff_values

['lig_vol',
 'pock_vol',
 'nb_AS',
 'mean_as_ray',
 'mean_as_solv_acc',
 'apol_as_prop',
 'mean_loc_hyd_dens',
 'hydrophobicity_score',
 'volume_score',
 'polarity_score',
 'charge_score',
 'flex',
 'prop_polar_atm',
 'as_density',
 'as_max_dst',
 'convex_hull_volume',
 'surf_pol_vdw14',
 'surf_pol_vdw22',
 'surf_apol_vdw14',
 'surf_apol_vdw22',
 'n_abpa',
 'ALA',
 'ARG',
 'ASN',
 'ASP',
 'CYS',
 'GLN',
 'GLU',
 'GLY',
 'HIS',
 'ILE',
 'LEU',
 'LYS',
 'MET',
 'PHE',
 'PRO',
 'SER',
 'THR',
 'TRP',
 'TYR',
 'VAL']

In [39]:
train = pd.read_parquet('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/atomica_ligand/masif_ligand_pdbs_12A_pocket_only_train.parquet')
train_pdbs = set(train['id'].str[:4].tolist())

val = pd.read_parquet('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/atomica_ligand/masif_ligand_pdbs_12A_pocket_only_val.parquet')
val_pdbs = set(val['id'].str[:4].tolist())

test = pd.read_parquet('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/atomica_ligand/masif_ligand_pdbs_12A_pocket_only_test.parquet')
test_pdbs = set(test['id'].str[:4].tolist())

assert len(train_pdbs & val_pdbs) == 0
assert len(train_pdbs & test_pdbs) == 0
assert len(val_pdbs & test_pdbs) == 0

In [41]:
df['split'] = df['pdb'].map(lambda x: 'train' if x in train_pdbs else 'val' if x in val_pdbs else 'test')

In [45]:
df['id'] = df['pdb'] + '_' + df['lig']
assert df['id'].nunique() == len(df)

In [42]:
df['split'].value_counts()

split
train    1039
test      289
val       117
Name: count, dtype: int64

In [48]:
df[['id','lig','split'] + cols_with_diff_values].to_csv('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/MASIF/dpocket/dpocket_out_4A_exp_with_split.csv', index=False)

# Test results

Whether or not the one hot AA features are used does not affect the results.

In [ ]:
results = pd.read_csv('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/MASIF/dpocket/dpocket_mlp_predictions.csv')
results = results[results['split'] == 'test']
(results['pred_label'] == results['true_label']).mean()

0.7577854671280276

In [78]:
results = pd.read_csv('/n/holylfs06/LABS/mzitnik_lab/Lab/afang/ATOMICA/baselines/MASIF/dpocket/dpocket_mlp_predictions_no_aa.csv')
results = results[results['split'] == 'test']
(results['pred_label'] == results['true_label']).mean()

0.7577854671280276